# 🎨 Combining Visualizations

**Difficulty:** 🔴 Hard  
**Prerequisites:** All previous visualization tutorials  
**Time:** 35-40 minutes  

---

## What You'll Learn

- How to create multi-panel figures combining different views
- How to use matplotlib subplots with maialib
- How to integrate plotly subplots for interactive dashboards
- How to create publication-quality figures
- How to export visualizations for papers and presentations
- How to build custom analysis dashboards

## Why This Matters

Single visualizations are useful, but **combining multiple views** reveals deeper insights:

- **See connections**: How does pitch relate to harmony? Density to orchestration?
- **Tell stories**: Multi-panel figures guide the reader through your analysis
- **Comprehensive analysis**: Present complete picture in one figure
- **Professional presentation**: Publication-ready figures for papers
- **Interactive exploration**: Dashboards for deep musical exploration

Essential for:
- Researchers writing papers
- Presenters creating compelling talks
- Teachers building teaching materials
- Composers analyzing reference works

---

## Step 1: Setup and Import

In [ ]:
# Import all necessary libraries
import maialib as ml
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print(f"✅ Maialib {ml.__version__} loaded successfully!")
print("🎨 Ready to create comprehensive visualizations!")

---

## Step 2: Simple Multi-Panel Figure with Matplotlib

Let's start by combining pitch envelope and note density in one figure.

In [ ]:
# Load a score
bach = ml.Score('Bach_Cello_Suite_1.mxl')

# Convert to DataFrame
df = bach.toDataFrame()

# Calculate statistics by measure
measure_stats = df.groupby('measure').agg({
    'midi': ['min', 'max', 'mean', 'count'],
    'duration': 'sum'
}).reset_index()

measure_stats.columns = ['measure', 'min_pitch', 'max_pitch', 'avg_pitch', 'note_count', 'total_duration']

# Create multi-panel figure
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Pitch envelope (min, max, average)
axes[0].fill_between(measure_stats['measure'], 
                       measure_stats['min_pitch'], 
                       measure_stats['max_pitch'],
                       alpha=0.3, color='steelblue', label='Pitch Range')
axes[0].plot(measure_stats['measure'], measure_stats['avg_pitch'], 
             color='darkblue', linewidth=2, label='Average Pitch')
axes[0].set_ylabel('MIDI Pitch', fontsize=12, fontweight='bold')
axes[0].set_title('Bach Cello Suite No. 1 - Comprehensive Analysis', 
                  fontsize=14, fontweight='bold', pad=20)
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Panel 2: Note density
axes[1].bar(measure_stats['measure'], measure_stats['note_count'], 
            color='orange', alpha=0.7, edgecolor='darkorange')
axes[1].set_ylabel('Notes per Measure', fontsize=12, fontweight='bold')
axes[1].axhline(measure_stats['note_count'].mean(), 
                color='red', linestyle='--', linewidth=2, label='Average')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Panel 3: Rhythmic density (total duration)
axes[2].fill_between(measure_stats['measure'], measure_stats['total_duration'],
                      alpha=0.6, color='green', edgecolor='darkgreen', linewidth=1.5)
axes[2].set_ylabel('Total Duration\n(quarter notes)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Measure Number', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Three-panel analysis created!")
print("Notice how all three metrics show related patterns.")

**Reading Multi-Panel Figures:**

**Look for correlations between panels**:
- When pitch range widens (Panel 1), do we see more notes (Panel 2)?
- When rhythmic density increases (Panel 3), does pitch change?
- Do all three metrics peak at the same moments (climax points)?

**In Bach's Cello Suite:**
- **Consistent note count** (Panel 2) = steady 16th note figuration
- **Variable pitch range** (Panel 1) = arpeggios of different chords
- **Stable duration** (Panel 3) = constant rhythmic pattern

This reveals Bach's technique: **constant rhythm with varying pitch** = hypnotic flow!

---

## Step 3: Interactive Multi-Panel Dashboard with Plotly

Now let's create an interactive dashboard that updates when you hover.

In [ ]:
# Load a richer orchestral score
beethoven = ml.Score('Beethoven_Symphony_5_mov_1.xml')

# Get comprehensive data
df = beethoven.toDataFrame()

# Calculate measure-level statistics
measure_stats = df.groupby('measure').agg({
    'midi': ['min', 'max', 'mean', 'std'],
    'pitch': 'count',
    'duration': ['sum', 'mean']
}).reset_index()

measure_stats.columns = ['measure', 'min_pitch', 'max_pitch', 'avg_pitch', 'pitch_std', 
                          'note_count', 'total_duration', 'avg_duration']

# Calculate pitch range
measure_stats['pitch_range'] = measure_stats['max_pitch'] - measure_stats['min_pitch']

# Get chords data
chords = beethoven.getChords()
chord_sizes = [c.size() for c in chords]

# Create plotly subplot figure
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=(
        'Pitch Envelope (Range and Average)',
        'Note Density per Measure',
        'Pitch Range (Texture Width)',
        'Pitch Variety (Standard Deviation)'
    ),
    vertical_spacing=0.08,
    specs=[[{"secondary_y": False}],
           [{"secondary_y": False}],
           [{"secondary_y": False}],
           [{"secondary_y": False}]]
)

# Row 1: Pitch envelope
fig.add_trace(
    go.Scatter(x=measure_stats['measure'], y=measure_stats['avg_pitch'],
               mode='lines', name='Average Pitch', line=dict(color='steelblue', width=2)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=measure_stats['measure'], y=measure_stats['max_pitch'],
               mode='lines', name='Max Pitch', line=dict(color='lightblue', width=1),
               fill=None),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=measure_stats['measure'], y=measure_stats['min_pitch'],
               mode='lines', name='Min Pitch', line=dict(color='lightblue', width=1),
               fill='tonexty', fillcolor='rgba(70, 130, 180, 0.2)'),
    row=1, col=1
)

# Row 2: Note density
fig.add_trace(
    go.Bar(x=measure_stats['measure'], y=measure_stats['note_count'],
           name='Note Count', marker_color='orange'),
    row=2, col=1
)

# Row 3: Pitch range
fig.add_trace(
    go.Scatter(x=measure_stats['measure'], y=measure_stats['pitch_range'],
               mode='lines', name='Pitch Range', 
               line=dict(color='green', width=2),
               fill='tozeroy', fillcolor='rgba(0, 128, 0, 0.2)'),
    row=3, col=1
)

# Row 4: Pitch variety (std dev)
fig.add_trace(
    go.Scatter(x=measure_stats['measure'], y=measure_stats['pitch_std'],
               mode='lines', name='Pitch Std Dev',
               line=dict(color='crimson', width=2),
               fill='tozeroy', fillcolor='rgba(220, 20, 60, 0.2)'),
    row=4, col=1
)

# Update layout
fig.update_xaxes(title_text="Measure Number", row=4, col=1)
fig.update_yaxes(title_text="MIDI", row=1, col=1)
fig.update_yaxes(title_text="Count", row=2, col=1)
fig.update_yaxes(title_text="Semitones", row=3, col=1)
fig.update_yaxes(title_text="Std Dev", row=4, col=1)

fig.update_layout(
    height=1000,
    title_text=f"Interactive Analysis Dashboard: {beethoven.getWorkTitle()}",
    showlegend=True,
    hovermode='x unified'
)

fig.show()

print("\n📊 Interactive dashboard created!")
print("Hover over any panel to see values across all metrics simultaneously.")

**Using Interactive Dashboards:**

**The power of `hovermode='x unified'`**:
- Hover over ANY panel
- See values for ALL metrics at that moment
- Instantly discover correlations

**Questions to explore by hovering**:
1. When pitch range is widest, is note density also high?
2. Do high pitch variety (std dev) moments correspond to climaxes?
3. Are there moments of high density but low range (close harmony)?
4. Where do all four metrics peak simultaneously?

**In Beethoven Symphony No. 5:**
- **Opening** (mm. 1-5): High density, wide range, high variety = dramatic statement
- **Transitions**: Low density, narrow range = chamber texture
- **Development**: Variable in all metrics = instability
- **Climaxes**: All four metrics spike together!

---

## Step 4: Comprehensive Score Analysis Figure

Let's create a publication-quality figure combining 6 different analysis views.

In [ ]:
def create_comprehensive_analysis(score, title=None):
    """
    Create a comprehensive 6-panel analysis of a musical score.
    """
    
    # Get data
    df = score.toDataFrame()
    chords = score.getChords()
    
    # Calculate statistics
    measure_stats = df.groupby('measure').agg({
        'midi': ['min', 'max', 'mean', 'std', 'count'],
        'duration': 'sum'
    }).reset_index()
    
    measure_stats.columns = ['measure', 'min_pitch', 'max_pitch', 'avg_pitch', 
                              'pitch_std', 'note_count', 'total_duration']
    measure_stats['pitch_range'] = measure_stats['max_pitch'] - measure_stats['min_pitch']
    
    # Chord data
    chord_sizes = [c.size() for c in chords if c.size() > 0]
    
    # Pitch class distribution
    pitch_class_counts = df['pitch_class'].value_counts().sort_index()
    
    # Create figure
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    # Panel 1: Pitch envelope
    ax1 = fig.add_subplot(gs[0, :])
    ax1.fill_between(measure_stats['measure'], 
                      measure_stats['min_pitch'], 
                      measure_stats['max_pitch'],
                      alpha=0.3, color='steelblue')
    ax1.plot(measure_stats['measure'], measure_stats['avg_pitch'], 
             color='darkblue', linewidth=2.5)
    ax1.set_ylabel('MIDI Pitch', fontsize=11, fontweight='bold')
    ax1.set_title('A. Pitch Envelope', fontsize=12, fontweight='bold', loc='left')
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: Note density
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.bar(measure_stats['measure'], measure_stats['note_count'], 
            color='orange', alpha=0.7)
    ax2.axhline(measure_stats['note_count'].mean(), 
                color='red', linestyle='--', linewidth=2)
    ax2.set_ylabel('Notes/Measure', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Measure', fontsize=11, fontweight='bold')
    ax2.set_title('B. Note Density', fontsize=12, fontweight='bold', loc='left')
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: Pitch range
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.fill_between(measure_stats['measure'], measure_stats['pitch_range'],
                      alpha=0.6, color='green')
    ax3.set_ylabel('Range (semitones)', fontsize=11, fontweight='bold')
    ax3.set_xlabel('Measure', fontsize=11, fontweight='bold')
    ax3.set_title('C. Pitch Range', fontsize=12, fontweight='bold', loc='left')
    ax3.grid(True, alpha=0.3)
    
    # Panel 4: Chord size distribution
    ax4 = fig.add_subplot(gs[2, 0])
    if chord_sizes:
        ax4.hist(chord_sizes, bins=range(1, max(chord_sizes)+2), 
                 color='purple', alpha=0.7, edgecolor='black')
        ax4.set_ylabel('Frequency', fontsize=11, fontweight='bold')
        ax4.set_xlabel('Chord Size (notes)', fontsize=11, fontweight='bold')
        ax4.set_title('D. Chord Size Distribution', fontsize=12, fontweight='bold', loc='left')
        ax4.grid(True, alpha=0.3, axis='y')
    
    # Panel 5: Pitch class distribution
    ax5 = fig.add_subplot(gs[2, 1])
    pitch_classes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    colors_pc = ['steelblue' if pc in pitch_class_counts.index else 'lightgray' 
                 for pc in pitch_classes]
    values = [pitch_class_counts.get(pc, 0) for pc in pitch_classes]
    
    ax5.bar(pitch_classes, values, color=colors_pc, edgecolor='black')
    ax5.set_ylabel('Note Count', fontsize=11, fontweight='bold')
    ax5.set_xlabel('Pitch Class', fontsize=11, fontweight='bold')
    ax5.set_title('E. Pitch Class Distribution', fontsize=12, fontweight='bold', loc='left')
    ax5.grid(True, alpha=0.3, axis='y')
    plt.setp(ax5.xaxis.get_majorticklabels(), rotation=45)
    
    # Overall title
    if title is None:
        title = f"{score.getComposer()}: {score.getWorkTitle()}"
    
    fig.suptitle(f'Comprehensive Musical Analysis\n{title}', 
                 fontsize=16, fontweight='bold', y=0.98)
    
    return fig

# Create comprehensive analysis
chopin = ml.Score('Chopin_Fantasie_Impromptu.mxl')
fig = create_comprehensive_analysis(chopin)
plt.show()

print("\n📊 Comprehensive 5-panel analysis created!")
print("This figure could be included in a research paper or presentation.")

**Publication-Quality Figure Elements:**

**Essential components**:
1. **Clear panel labels** (A, B, C, D, E) for reference in text
2. **Descriptive titles** for each panel
3. **Axis labels with units** (MIDI pitch, semitones, etc.)
4. **Consistent style** across all panels
5. **Overall title** with composer and work
6. **High resolution** (for print: 300+ DPI)

**To save for publication**:
```python
fig.savefig('chopin_analysis.png', dpi=300, bbox_inches='tight')
fig.savefig('chopin_analysis.pdf', bbox_inches='tight')  # Vector format
```

**In your paper**, reference like:
> "Figure 1A shows the pitch envelope, revealing an arch-shaped contour typical of Romantic piano music. The note density (Fig. 1B) remains consistently high, reflecting Chopin's virtuosic writing..."

---

## Step 5: Comparative Analysis Dashboard

Let's compare multiple pieces side-by-side in one comprehensive figure.

In [ ]:
# Load multiple scores for comparison
scores = {
    'Bach': ml.Score('Bach_Cello_Suite_1.mxl'),
    'Mozart': ml.Score('Mozart_Requiem_Introitus.mxl'),
    'Beethoven': ml.Score('Beethoven_Symphony_5_mov_1.xml'),
    'Chopin': ml.Score('Chopin_Fantasie_Impromptu.mxl'),
}

# Calculate summary statistics for each
comparison_data = []

for name, score in scores.items():
    df = score.toDataFrame()
    chords = score.getChords()
    
    # Measure-level stats
    measure_stats = df.groupby('measure').agg({
        'midi': ['mean', 'std'],
        'pitch': 'count'
    })
    
    chord_sizes = [c.size() for c in chords if c.size() > 0]
    
    comparison_data.append({
        'Composer': name,
        'Avg Pitch': df['midi'].mean(),
        'Pitch Range': df['midi'].max() - df['midi'].min(),
        'Avg Notes/Measure': measure_stats[('pitch', 'count')].mean(),
        'Pitch Variety': df['midi'].std(),
        'Avg Chord Size': np.mean(chord_sizes) if chord_sizes else 0,
        'Total Notes': len(df)
    })

df_comparison = pd.DataFrame(comparison_data)

print("Comparative Analysis:")
print(df_comparison.to_string(index=False))

# Create radar chart comparison
fig = go.Figure()

# Normalize metrics to 0-100 scale for radar chart
metrics = ['Avg Pitch', 'Pitch Range', 'Avg Notes/Measure', 'Pitch Variety', 'Avg Chord Size']
colors = ['steelblue', 'orange', 'green', 'crimson']

for i, (_, row) in enumerate(df_comparison.iterrows()):
    # Normalize values
    values = [
        row['Avg Pitch'] / df_comparison['Avg Pitch'].max() * 100,
        row['Pitch Range'] / df_comparison['Pitch Range'].max() * 100,
        row['Avg Notes/Measure'] / df_comparison['Avg Notes/Measure'].max() * 100,
        row['Pitch Variety'] / df_comparison['Pitch Variety'].max() * 100,
        row['Avg Chord Size'] / df_comparison['Avg Chord Size'].max() * 100 if df_comparison['Avg Chord Size'].max() > 0 else 0,
    ]
    
    fig.add_trace(go.Scatterpolar(
        r=values + [values[0]],  # Close the polygon
        theta=metrics + [metrics[0]],
        fill='toself',
        name=row['Composer'],
        line_color=colors[i]
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100]
        )
    ),
    showlegend=True,
    title="Comparative Musical Characteristics (Normalized)",
    height=600
)

fig.show()

print("\n📊 Radar chart comparison created!")
print("Each composer's musical 'fingerprint' is shown as a polygon.")
print("Larger areas = more extreme values in those characteristics.")

**Interpreting Radar Charts:**

**What radar charts show**:
- **Overall size**: More extreme/distinctive characteristics
- **Shape**: Unique "fingerprint" of each composer's style
- **Overlap**: Similar characteristics between composers
- **Spikes**: Areas where a composer excels or differs

**Expected patterns**:

**Bach**:
- Moderate pitch range (cello solo)
- Moderate chord size (implied harmony)
- Consistent note density

**Mozart**:
- Wide pitch range (orchestra)
- Large chord size (chorus + orchestra)
- High note count (many parts)

**Beethoven**:
- Very wide pitch range (dramatic)
- High variety (dynamic contrasts)
- Large ensemble

**Chopin**:
- Wide pitch range (full piano keyboard)
- High note density (virtuosic)
- Complex harmony (large chords)

---

## Step 6: Exporting for Different Formats

Let's learn how to export our visualizations for different purposes.

In [ ]:
# Create a sample figure
score = ml.Score('Bach_Prelude_1_BWV_846.mxl')
fig = create_comprehensive_analysis(score)

# Export for different purposes

# 1. For print publication (high resolution PNG)
fig.savefig('analysis_print.png', dpi=300, bbox_inches='tight', facecolor='white')
print("✅ Saved high-res PNG for print (300 DPI)")

# 2. For web/presentation (lower resolution, smaller file)
fig.savefig('analysis_web.png', dpi=150, bbox_inches='tight', facecolor='white')
print("✅ Saved web-optimized PNG (150 DPI)")

# 3. For LaTeX papers (vector PDF, scales perfectly)
fig.savefig('analysis_paper.pdf', bbox_inches='tight', facecolor='white')
print("✅ Saved vector PDF for LaTeX")

# 4. For editing in Illustrator (vector SVG)
fig.savefig('analysis_editable.svg', bbox_inches='tight', facecolor='white')
print("✅ Saved editable SVG")

print("\n📁 Files saved:")
print("  - analysis_print.png (high-res, for journals)")
print("  - analysis_web.png (smaller, for websites/slides)")
print("  - analysis_paper.pdf (vector, for papers)")
print("  - analysis_editable.svg (vector, for editing)")

plt.close(fig)

# For Plotly interactive figures
# plotly_fig.write_html('interactive_analysis.html')  # Self-contained HTML
# plotly_fig.write_image('analysis_plotly.png', width=1200, height=800)  # Static image

**Export Format Guide:**

| Format | Use Case | Pros | Cons |
|--------|----------|------|------|
| **PNG (300 DPI)** | Print journals, books | High quality, universal | Large files |
| **PNG (150 DPI)** | Websites, presentations | Smaller files | Lower quality |
| **PDF** | LaTeX papers, printing | Vector (scales perfectly) | Larger than PNG |
| **SVG** | Further editing | Editable, scalable | Not all software supports |
| **HTML** | Interactive web | Fully interactive | Requires browser |

**Best practices**:
- **Journal submission**: PDF (vector) or high-res PNG (300+ DPI)
- **Conference talk**: PNG 150 DPI (loads fast)
- **Website/blog**: PNG 100-150 DPI or interactive HTML
- **Poster**: PDF or PNG 300 DPI
- **Social media**: PNG 72-100 DPI (small files)

---

## 🎯 Practice Exercises

### Exercise 1: Create Your Own Multi-Panel Figure

Choose any score and create a 3-panel figure showing:
1. Pitch envelope
2. Note density
3. Any third metric of your choice

Make it publication-quality with proper labels and title.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Click to see solution</summary>

```python
# Load score
score = ml.Score('Dvorak_Symphony_9_mov_4.mxl')
df = score.toDataFrame()

# Calculate stats
measure_stats = df.groupby('measure').agg({
    'midi': ['min', 'max', 'mean'],
    'pitch': 'count',
    'duration': 'mean'
}).reset_index()

measure_stats.columns = ['measure', 'min', 'max', 'avg', 'count', 'avg_dur']

# Create figure
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Pitch
axes[0].fill_between(measure_stats['measure'], measure_stats['min'], measure_stats['max'], alpha=0.3)
axes[0].plot(measure_stats['measure'], measure_stats['avg'], linewidth=2)
axes[0].set_title('A. Pitch Envelope', loc='left', fontweight='bold')
axes[0].set_ylabel('MIDI Pitch')

# Panel 2: Density
axes[1].bar(measure_stats['measure'], measure_stats['count'], color='orange', alpha=0.7)
axes[1].set_title('B. Note Density', loc='left', fontweight='bold')
axes[1].set_ylabel('Notes/Measure')

# Panel 3: Average duration
axes[2].plot(measure_stats['measure'], measure_stats['avg_dur'], color='green', linewidth=2)
axes[2].set_title('C. Average Note Duration', loc='left', fontweight='bold')
axes[2].set_ylabel('Duration')
axes[2].set_xlabel('Measure')

fig.suptitle(f'{score.getComposer()}: {score.getWorkTitle()}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
```
</details>

---

### Exercise 2: Build an Interactive Dashboard

Create a Plotly subplot figure with at least 4 different views of a score. Use `hovermode='x unified'` for interactivity.

In [ ]:
# YOUR CODE HERE


### Exercise 3: Comparative Radar Chart

Compare 3 different pieces on a radar chart with at least 5 different musical metrics.

In [ ]:
# YOUR CODE HERE


---

## ✅ Summary

Congratulations! You've mastered combining visualizations. You can now:

- ✅ Create multi-panel figures with matplotlib
- ✅ Build interactive dashboards with Plotly subplots
- ✅ Generate publication-quality comprehensive analyses
- ✅ Compare multiple pieces with radar charts
- ✅ Export visualizations in appropriate formats
- ✅ Design figures for papers, presentations, and web

## Key Concepts

- **Multi-panel figures** reveal relationships between metrics
- **Interactive dashboards** enable deep exploration
- **Publication quality** requires proper labels, titles, and resolution
- **Comparative visualizations** show differences between pieces
- **Format matters**: PDF for papers, PNG for web, HTML for interactive
- **Unified hover** in Plotly connects all panels simultaneously

## 🎯 Next Steps

Apply these visualization skills to analysis:

1. **[Harmonic Complexity Analysis](../05_analysis/01_harmonic_complexity_analysis.ipynb)** - Deep harmonic studies
2. **[Dissonance Curves](../05_analysis/02_dissonance_curves_analysis.ipynb)** - Psychoacoustic analysis
3. **[Corpus Statistics](../05_analysis/03_corpus_statistics.ipynb)** - Multi-score comparative studies
4. **[Form and Structure](../05_analysis/05_form_and_structure_analysis.ipynb)** - Use visualizations to identify sections

Or create your own analyses:

- Research papers with comprehensive figures
- Interactive online music analysis tools
- Teaching materials for music theory classes
- Program notes with visualizations

---

<div align="center">

**Tell the complete story! 🎨**

*Combined visualizations transform isolated observations into comprehensive narratives about musical structure and meaning*

</div>